# SQL feature engineering (SQLite)

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Write joins, aggregations and window functions
- Build point-in-time-correct features
- Spot leakage

In [ ]:
import sqlite3, pandas as pd, numpy as np
con = sqlite3.connect(':memory:')
rng = np.random.default_rng(3)
customers = pd.DataFrame({'customer_id': range(1, 201), 'country': rng.choice(['IN', 'GB', 'US'], 200)})
orders = pd.DataFrame({'order_id': range(1, 2001), 'customer_id': rng.integers(1, 201, 2000), 'amount': rng.gamma(2, 20, 2000).round(2), 'created_at': pd.to_datetime('2026-01-01') + pd.to_timedelta(rng.integers(0, 270, 2000), unit='D')})
customers.to_sql('customers', con, index=False); orders.assign(created_at=orders.created_at.dt.strftime('%Y-%m-%d')).to_sql('orders', con, index=False)
q = lambda s: pd.read_sql(s, con)

## 1. Join and aggregate

In [ ]:
q('''SELECT c.country, COUNT(o.order_id) AS orders, ROUND(AVG(o.amount), 2) AS avg_amount
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.country HAVING COUNT(o.order_id) > 0 ORDER BY orders DESC''')

## 2. Window functions

In [ ]:
q('''SELECT customer_id, order_id, created_at, amount,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY created_at) AS order_no,
       ROUND(AVG(amount) OVER (PARTITION BY customer_id ORDER BY created_at ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING), 2) AS avg_prev_3
FROM orders ORDER BY customer_id, created_at LIMIT 12''')

## 3. Point-in-time features for a prediction on 2026-07-01

In [ ]:
q('''SELECT c.customer_id,
  COUNT(o.order_id) AS orders_90d,
  COALESCE(SUM(o.amount), 0) AS spend_90d
FROM customers c LEFT JOIN orders o
  ON o.customer_id = c.customer_id
 AND o.created_at <  '2026-07-01'
 AND o.created_at >= '2026-04-02'
GROUP BY c.customer_id LIMIT 10''')

## Exercises
1. For each customer, find their largest order and its date.
2. Write the *leaky* version of section 3 (no date filter) and explain why it would inflate model accuracy.

In [ ]:
# your work here

## Solutions

In [ ]:
q('''SELECT customer_id, order_id, amount, created_at FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS r FROM orders) WHERE r = 1 LIMIT 10''')